# 数据管理：从多表合并到可复用工作流

拿到数据后，真正困难的往往不是最后一步模型估计，而是模型估计之前的三件事：多张表能不能合并，合并以后是否仍然保持正确的分析单位，清洗后的数据如何保存和复用。

本章把原来的「数据管理与组织」和「数据管理：格式、存储与工作流」合并为一个完整案例。主线如下：

$$
\text{source tables}\rightarrow \text{keys and grain}\rightarrow \text{analysis base table}\rightarrow \text{storage and workflow}
$$

本章不系统讲数据库理论，也不讨论服务器部署。重点是经管类实证研究中最常见的本地数据管理问题：

- 多来源、多频率的数据表如何判断主键和粒度；
- 为什么不能把 `firm-year` 表和 `firm-date` 表直接粗暴合并；
- 如何构造一张可用于后续建模的 `analysis_base`；
- 何时使用 CSV、Parquet、SQLite 和 DuckDB；
- 如何把数据文件、代码和说明文档组织成可复用工作流。

## 本章导读

课堂练习中，数据通常已经被整理成一张干净的 CSV 表。真实项目更常见的情况是：财务报表是一张表，日度行情是一批文件，宏观变量按月发布，公司公告是文本或元数据，行业分类又来自另一个来源。

此时，问题不再是「如何读入一张表」，而是「如何把多张表组织成一个可以持续维护的数据项目」。本章的核心判断可以压缩为三句话：

- 合并之前，先问清楚每张表的一行代表什么；
- 构造分析底表时，先确定目标粒度，再处理其他频率的数据；
- 保存数据时，不同格式服务于不同任务，不存在一种格式适合所有场景。

本章生成的所有演示文件默认保存在 `_demo_data_manage/` 文件夹中，避免误改本地已有的 `data/`、`data_raw/` 或 `figs/` 文件。

## 当前讲义文件夹的一个整理建议

从当前 `data_manage/` 文件夹结构看，主要问题不是文件太多，而是同类文件分散在多个位置。例如，既有 `data/`，也有 `data_raw/`；既有当前讲义文件，也有旧版 notebook 和作废存档；图形文件、转换脚本和 HTML 表格都放在 `figs/` 下。这种结构在赶课件时很常见，但后续维护会比较麻烦。

建议合并本章以后，逐步整理为如下结构：

```text
data_manage/
├── lecture_data_management.ipynb        # 合并后的本章讲义
├── lecture_data_management.md           # 如需保留，可由 notebook 导出
├── codes/                               # 可复用代码和辅助脚本
├── data/                                # 推荐的统一数据根目录
│   ├── raw/                             # 原始数据，原则上不手工修改
│   ├── processed/                       # 清洗后的标准化数据
│   └── outputs/                         # 分析结果、中间表和导出数据
├── figs/                                # 本章插图
└── archive/                             # 旧版文件和作废材料
```

需要说明的是，如果前面章节的代码已经大量使用 `data_raw/`，不建议立刻大规模改路径。更稳妥的做法是：本章文字中介绍推荐结构，代码中保持兼容；等整本书统一重构时，再把 `data_raw/` 迁移到 `data/raw/`。

## 环境准备与演示目录

下面代码只会创建 `_demo_data_manage/` 演示目录。这个目录可以反复删除和重新生成，不影响课程文件夹中已有的真实数据。

In [1]:
# 基础包
# 说明：本章尽量只依赖 Python 标准库、pandas 和 numpy。
# Parquet 和 DuckDB 相关部分会自动检查依赖；如果本地没有安装，会给出提示而不是中断运行。

from pathlib import Path
import sqlite3
import shutil
import os
import textwrap

import numpy as np
import pandas as pd
from IPython.display import display

# 演示目录。为避免误改真实课程数据，本章所有输出都写入这个目录。
DEMO_ROOT = Path('_demo_data_manage')
DATA_RAW = DEMO_ROOT / 'data' / 'raw'
DATA_PROCESSED = DEMO_ROOT / 'data' / 'processed'
DATA_OUTPUTS = DEMO_ROOT / 'data' / 'outputs'
DOCS_DIR = DEMO_ROOT / 'docs'

# 如果你希望每次重新执行都得到干净结果，可以取消下一行注释。
# shutil.rmtree(DEMO_ROOT, ignore_errors=True)

for path in [DATA_RAW, DATA_PROCESSED, DATA_OUTPUTS, DOCS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f'演示目录：{DEMO_ROOT.resolve()}')
for path in [DATA_RAW, DATA_PROCESSED, DATA_OUTPUTS, DOCS_DIR]:
    print('-', path)

演示目录：/mnt/data/_demo_data_manage
- _demo_data_manage/data/raw
- _demo_data_manage/data/processed
- _demo_data_manage/data/outputs
- _demo_data_manage/docs


## 一个金融研究中的多表场景

设想我们要构造一张 `firm-year` 层面的分析底表。手头有三类数据：

| 表名 | 粒度 | 主键 | 典型变量 | 用途 |
|---|---:|---|---|---|
| `basic_info` | firm | `stock_code` | 公司名称、行业 | 补充公司层面属性 |
| `fin_ratio` | firm-year | `stock_code, year` | ROE、资产负债率 | 年度财务指标 |
| `daily_ret` | firm-date | `stock_code, trade_date` | 日收益率、换手率 | 构造年度市场特征 |

最终目标不是把所有字段机械拼到一起，而是构造 `analysis_base`：一张主键为 `(stock_code, year)` 的分析底表。后续回归、画图和建模都以这张表为基础。

In [2]:
# 构造三张不同粒度的数据表
# 这组数据很小，目的是让主键、粒度和合并逻辑看得清楚。

basic_info = pd.DataFrame({
    'stock_code': ['000001', '000002', '000003'],
    'firm_name': ['平安银行', '万科A', '国农科技'],
    'industry_name': ['银行', '房地产', '医药生物']
})

fin_ratio = pd.DataFrame({
    'stock_code': ['000001', '000001', '000002', '000002', '000003', '000003'],
    'year': [2022, 2023, 2022, 2023, 2022, 2023],
    'roe': [0.098, 0.105, 0.072, 0.068, 0.041, 0.055],
    'leverage': [0.915, 0.918, 0.772, 0.781, 0.432, 0.446]
})

trade_date = pd.to_datetime([
    '2023-01-03', '2023-01-04', '2023-01-05',
    '2023-01-03', '2023-01-04', '2023-01-05',
    '2023-01-03', '2023-01-04', '2023-01-05'
])

daily_ret = pd.DataFrame({
    'stock_code': ['000001'] * 3 + ['000002'] * 3 + ['000003'] * 3,
    'trade_date': trade_date,
    'ret': [0.010, -0.004, 0.006, 0.008, -0.002, 0.003, -0.005, 0.011, 0.002],
    'turnover': [0.021, 0.018, 0.025, 0.016, 0.015, 0.017, 0.010, 0.012, 0.014]
})

print('basic_info：公司基本信息表')
display(basic_info)

print('fin_ratio：年度财务指标表')
display(fin_ratio)

print('daily_ret：日度收益率表')
display(daily_ret)

basic_info：公司基本信息表


,stock_code,firm_name,industry_name
0,000001,平安银行,银行
1,000002,万科A,房地产
2,000003,国农科技,医药生物


fin_ratio：年度财务指标表


,stock_code,year,roe,leverage
0,000001,2022,0.098,0.915
1,000001,2023,0.105,0.918
2,000002,2022,0.072,0.772
3,000002,2023,0.068,0.781
4,000003,2022,0.041,0.432
5,000003,2023,0.055,0.446


daily_ret：日度收益率表


,stock_code,trade_date,ret,turnover
0,000001,2023-01-03,0.010,0.021
1,000001,2023-01-04,-0.004,0.018
2,000001,2023-01-05,0.006,0.025
3,000002,2023-01-03,0.008,0.016
4,000002,2023-01-04,-0.002,0.015
5,000002,2023-01-05,0.003,0.017
6,000003,2023-01-03,-0.005,0.010
7,000003,2023-01-04,0.011,0.012
8,000003,2023-01-05,0.002,0.014


## 主键与粒度：合并前先问清楚三件事

多表合并之前，至少要问三个问题：

- 这张表的一行代表什么？
- 哪些变量或变量组合可以唯一识别一行？
- 最终分析底表的目标粒度是什么？

主键可以理解为唯一识别一行数据的变量或变量组合。若 $K$ 表示候选主键，$N$ 表示数据表行数，则一个直观检查是：

$$
\operatorname{unique}(K)=N
$$

如果候选主键的唯一取值数小于行数，就说明该主键不能唯一识别一行数据。此时继续合并，很可能导致重复匹配或样本膨胀。

In [3]:
def check_key(df, keys, table_name):
    """
    检查给定 keys 是否能唯一识别 df 中的一行。

    参数
    ----
    df : pandas.DataFrame
        待检查的数据表。
    keys : list[str]
        候选主键变量名。
    table_name : str
        数据表名称，用于打印提示信息。
    """
    n_rows = len(df)
    n_unique = df[keys].drop_duplicates().shape[0]
    has_dup = df.duplicated(subset=keys).any()

    print(f'{table_name} 的行数：{n_rows}')
    print(f'{table_name} 的唯一主键数：{n_unique}')
    print(f'{table_name} 是否存在重复主键：{has_dup}')
    print('-' * 50)

check_key(basic_info, ['stock_code'], 'basic_info')
check_key(fin_ratio, ['stock_code', 'year'], 'fin_ratio')
check_key(daily_ret, ['stock_code', 'trade_date'], 'daily_ret')

basic_info 的行数：3
basic_info 的唯一主键数：3
basic_info 是否存在重复主键：False
--------------------------------------------------
fin_ratio 的行数：6
fin_ratio 的唯一主键数：6
fin_ratio 是否存在重复主键：False
--------------------------------------------------
daily_ret 的行数：9
daily_ret 的唯一主键数：9
daily_ret 是否存在重复主键：False
--------------------------------------------------


## 一个典型错误：忽略粒度直接合并

`fin_ratio` 的粒度是 `firm-year`，`daily_ret` 的粒度是 `firm-date`。如果只按 `stock_code` 直接合并，代码可以正常运行，但得到的表已经不再是 `firm-year` 数据。

这类错误的危险之处在于：它通常不会报错，反而会生成一张看似更大的表。样本量变多并不代表信息变多，很多时候只是分析单位被错误展开。

In [4]:
# 错误示例：忽略 year 和 trade_date 的粒度差异，只按 stock_code 合并
wrong_merge = fin_ratio.merge(daily_ret, on='stock_code', how='left')

print('fin_ratio 原始行数：', len(fin_ratio))
print('wrong_merge 合并后行数：', len(wrong_merge))
print('wrong_merge 中 (stock_code, year) 是否重复：',
      wrong_merge.duplicated(['stock_code', 'year']).any())

display(wrong_merge.head(10))

fin_ratio 原始行数： 6
wrong_merge 合并后行数： 18
wrong_merge 中 (stock_code, year) 是否重复： True


,stock_code,year,roe,leverage,trade_date,ret,turnover
0,000001,2022,0.098,0.915,2023-01-03,0.010,0.021
1,000001,2022,0.098,0.915,2023-01-04,-0.004,0.018
2,000001,2022,0.098,0.915,2023-01-05,0.006,0.025
3,000001,2023,0.105,0.918,2023-01-03,0.010,0.021
4,000001,2023,0.105,0.918,2023-01-04,-0.004,0.018
5,000001,2023,0.105,0.918,2023-01-05,0.006,0.025
6,000002,2022,0.072,0.772,2023-01-03,0.008,0.016
7,000002,2022,0.072,0.772,2023-01-04,-0.002,0.015
8,000002,2022,0.072,0.772,2023-01-05,0.003,0.017
9,000002,2023,0.068,0.781,2023-01-03,0.008,0.016


上面的结果说明，`wrong_merge` 的行数从 6 行膨胀到 18 行。问题不在于 `merge()` 函数，而在于合并条件没有尊重数据粒度。

对于以 `firm-year` 为目标粒度的研究，`daily_ret` 不能直接并入 `fin_ratio`。正确做法是先把日度数据聚合到年度层面，再按 `(stock_code, year)` 合并。

## 从高频数据到分析底表

若最终分析单位是 `firm-year`，日度收益率表需要先转换为年度特征。例如：

$$
\operatorname{ret\_mean}_{i,t}=\frac{1}{N_{i,t}}\sum_{d\in t}r_{i,d}
$$

$$
\operatorname{ret\_sd}_{i,t}=\sqrt{\frac{1}{N_{i,t}-1}\sum_{d\in t}\left(r_{i,d}-\operatorname{ret\_mean}_{i,t}\right)^{2}}
$$

这里的关键不是 `groupby()` 本身，而是先把 `firm-date` 表转换成 `firm-year` 表。只有当两张表处在同一目标粒度上，合并才有明确的经济含义。

In [5]:
# 正确做法：先把日度收益率聚合到 firm-year 层面
ret_yearly = daily_ret.copy()
ret_yearly['year'] = ret_yearly['trade_date'].dt.year

ret_yearly = (
    ret_yearly
    .groupby(['stock_code', 'year'], as_index=False)
    .agg(
        ret_mean=('ret', 'mean'),
        ret_sd=('ret', 'std'),
        turnover_mean=('turnover', 'mean'),
        n_trade_days=('ret', 'size')
    )
)

check_key(ret_yearly, ['stock_code', 'year'], 'ret_yearly')
display(ret_yearly)

ret_yearly 的行数：3
ret_yearly 的唯一主键数：3
ret_yearly 是否存在重复主键：False
--------------------------------------------------


,stock_code,year,ret_mean,ret_sd,turnover_mean,n_trade_days
0,000001,2023,0.004000,0.007211,0.021333,3
1,000002,2023,0.003000,0.005000,0.016000,3
2,000003,2023,0.002667,0.008021,0.012000,3


In [6]:
# 构造最终分析底表
# fin_ratio 是主表，因为它已经处在目标粒度 firm-year。
# ret_yearly 已经从 firm-date 聚合到 firm-year，可以按 (stock_code, year) 合并。
# basic_info 是 firm 层面的静态表，可以按 stock_code 合并。

analysis_base = (
    fin_ratio
    .merge(ret_yearly, on=['stock_code', 'year'], how='left')
    .merge(basic_info, on='stock_code', how='left')
)

check_key(analysis_base, ['stock_code', 'year'], 'analysis_base')
display(analysis_base)

analysis_base 的行数：6
analysis_base 的唯一主键数：6
analysis_base 是否存在重复主键：False
--------------------------------------------------


,stock_code,year,roe,leverage,ret_mean,ret_sd,turnover_mean,n_trade_days,firm_name,industry_name
0,000001,2022,0.098,0.915,NaN,NaN,NaN,NaN,平安银行,银行
1,000001,2023,0.105,0.918,0.004000,0.007211,0.021333,3.0,平安银行,银行
2,000002,2022,0.072,0.772,NaN,NaN,NaN,NaN,万科A,房地产
3,000002,2023,0.068,0.781,0.003000,0.005000,0.016000,3.0,万科A,房地产
4,000003,2022,0.041,0.432,NaN,NaN,NaN,NaN,国农科技,医药生物
5,000003,2023,0.055,0.446,0.002667,0.008021,0.012000,3.0,国农科技,医药生物


## 数据契约：让每张表先说清楚自己是什么

合并数据之前，应先写清楚每张表的「数据契约」。这里的契约不是法律文件，而是让数据表先说明自己是什么：粒度是什么，主键是什么，能否直接进入分析底表，需要先做什么转换。

| 表名 | 粒度 | 主键 | 能否直接并入 `analysis_base` | 处理方式 |
|---|---:|---|---|---|
| `basic_info` | firm | `stock_code` | 可以 | 按 `stock_code` 合并 |
| `fin_ratio` | firm-year | `stock_code, year` | 本身就是主表 | 作为分析底表基础 |
| `daily_ret` | firm-date | `stock_code, trade_date` | 不可以 | 先聚合为 `ret_yearly` |
| `ret_yearly` | firm-year | `stock_code, year` | 可以 | 按 `(stock_code, year)` 合并 |
| `analysis_base` | firm-year | `stock_code, year` | 最终结果 | 用于后续建模 |

这种表格可以放入 README 或数据字典。它的价值在于：未来自己或合作者再次打开项目时，不必重新猜测每张表的含义。

In [7]:
# 生成一张数据契约表，后续可以导出为 CSV 或 Markdown

data_contract = pd.DataFrame({
    'table_name': ['basic_info', 'fin_ratio', 'daily_ret', 'ret_yearly', 'analysis_base'],
    'grain': ['firm', 'firm-year', 'firm-date', 'firm-year', 'firm-year'],
    'key': ['stock_code', 'stock_code, year', 'stock_code, trade_date', 'stock_code, year', 'stock_code, year'],
    'role': [
        '公司层面静态属性',
        '年度财务指标主表',
        '日度市场数据原表',
        '由日度数据聚合得到的年度特征',
        '最终分析底表'
    ],
    'how_to_use': [
        '按 stock_code 合并',
        '作为分析底表基础',
        '先聚合到 firm-year',
        '按 stock_code, year 合并',
        '用于后续建模'
    ]
})

display(data_contract)

# 保存数据契约
contract_path = DOCS_DIR / 'data_contract.csv'
data_contract.to_csv(contract_path, index=False, encoding='utf-8-sig')
print(f'数据契约已保存：{contract_path}')

,table_name,grain,key,role,how_to_use
0,basic_info,firm,stock_code,公司层面静态属性,按 stock_code 合并
1,fin_ratio,firm-year,"stock_code, year",年度财务指标主表,作为分析底表基础
2,daily_ret,firm-date,"stock_code, trade_date",日度市场数据原表,先聚合到 firm-year
3,ret_yearly,firm-year,"stock_code, year",由日度数据聚合得到的年度特征,"按 stock_code, year 合并"
4,analysis_base,firm-year,"stock_code, year",最终分析底表,用于后续建模


数据契约已保存：_demo_data_manage/docs/data_contract.csv


## 文件格式选择：没有一种格式适合所有场景

构造好分析底表后，下一步才是选择存储方式。常见选择可以先用一张表判断：

| 场景 | 推荐方式 | 说明 |
|---|---|---|
| 小数据、人工查看、跨软件交换 | CSV / Excel | 容易打开，但类型信息弱 |
| 清洗后的标准化大表反复读取 | Parquet | 体积小、读取快、保留类型信息 |
| 多张结构化表需要反复关联 | SQLite | 一个 `.db` 文件管理多张表，适合本地关系查询 |
| 多个本地大文件需要直接 SQL 查询 | DuckDB | 适合分析型查询，可直接读 Parquet / CSV |

可以把这个选择理解为：CSV 适合给人看，Parquet 适合给程序反复读，SQLite 适合管理结构化关系表，DuckDB 适合直接查询本地分析文件。

![数据格式选择决策树](figs/data_manage_data_format_decision_tree.png)

## CSV 与 Parquet：从人工交换到机器读取

CSV 的优点是直观、通用、便于人工检查；局限是读写慢、文件大、类型信息弱。股票代码中的前导零、日期变量、缺失值标记，都可能在读入时出问题。

Parquet 是面向分析场景的列式存储格式。它通常具有三个优势：

- 可以保留较完整的字段类型信息；
- 可以只读取部分列，避免把整张表全部加载进内存；
- 文件体积通常小于 CSV，适合反复读取和中间结果保存。

需要说明的是，`pandas` 提供 `read_parquet()` 和 `to_parquet()` 接口，但本地通常需要安装 Parquet 引擎，常用的是 `pyarrow` 或 `fastparquet`。本课程建议安装：

```bash
pip install pyarrow
```

![CSV 与 Parquet 的存储方式比较](figs/data_manage_parquet_vs_csv_storage_format.png)

In [8]:
# 保存 CSV 和 Parquet，并比较文件大小
# 若本地没有安装 pyarrow 或 fastparquet，Parquet 部分会跳过。

csv_path = DATA_PROCESSED / 'analysis_base.csv'
parquet_path = DATA_PROCESSED / 'analysis_base.parquet'

analysis_base.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'CSV 已保存：{csv_path}')

try:
    analysis_base.to_parquet(parquet_path, index=False)
    has_parquet = True
    print(f'Parquet 已保存：{parquet_path}')
except ImportError as e:
    has_parquet = False
    print('当前环境没有可用的 Parquet 引擎。')
    print('如需运行 Parquet 示例，请先安装：pip install pyarrow')

csv_size = csv_path.stat().st_size / 1024
print(f'CSV 文件大小：{csv_size:.2f} KB')

if has_parquet:
    parquet_size = parquet_path.stat().st_size / 1024
    print(f'Parquet 文件大小：{parquet_size:.2f} KB')

CSV 已保存：_demo_data_manage/data/processed/analysis_base.csv
当前环境没有可用的 Parquet 引擎。
如需运行 Parquet 示例，请先安装：pip install pyarrow
CSV 文件大小：0.53 KB


In [9]:
# 读取示例：CSV 需要重新识别类型；Parquet 通常能更好地保留字段类型。

read_csv_demo = pd.read_csv(csv_path, dtype={'stock_code': str})
print('CSV 读入后的数据类型：')
print(read_csv_demo.dtypes)

if has_parquet:
    read_parquet_demo = pd.read_parquet(parquet_path)
    print('\nParquet 读入后的数据类型：')
    print(read_parquet_demo.dtypes)

CSV 读入后的数据类型：
stock_code        object
year               int64
roe              float64
leverage         float64
ret_mean         float64
ret_sd           float64
turnover_mean    float64
n_trade_days     float64
firm_name         object
industry_name     object
dtype: object


## Schema：类型也是数据契约的一部分

在 CSV 中，字段类型主要依赖读入时的推断和手工指定；在 Parquet 中，字段类型会随文件一起保存。对于金融数据，这一点很有价值。

例如，`stock_code` 不应被读成整数，否则 `000001` 会变成 `1`。这类错误不会改变变量名，却会破坏后续合并。类型契约的作用，就是减少这类隐蔽错误。

![Parquet 内部结构示意](figs/data_manage_parquet_internal_structure.png)

In [10]:
# 如果安装了 pyarrow，可以读取 Parquet 的 Schema。

if has_parquet:
    try:
        import pyarrow.parquet as pq
        schema = pq.read_schema(parquet_path)
        print(schema)
    except ImportError:
        print('已能写入 Parquet，但当前环境未安装 pyarrow，跳过 Schema 展示。')
else:
    print('未生成 Parquet 文件，跳过 Schema 展示。')

未生成 Parquet 文件，跳过 Schema 展示。


## SQLite：把多张结构化表放进一个数据库文件

Parquet 适合保存标准化大表，但它不是关系数据库。当一个项目中有多张结构化表，并且需要反复查询和连接时，可以考虑 SQLite。

SQLite 的特点是：

- Python 标准库内置 `sqlite3`，不需要安装数据库服务器；
- 整个数据库就是一个 `.db` 或 `.sqlite` 文件；
- 支持 SQL 查询、连接、聚合和索引；
- 适合本地单人项目或课程演示。

需要说明的是，`pandas.DataFrame.to_sql()` 只是把 DataFrame 写成数据库表，并不会自动把 `(stock_code, year)` 设置为数据库层面的主键约束。如果需要数据库强制主键约束，应先用 `CREATE TABLE ... PRIMARY KEY (...)` 建表，再插入数据。

![SQLite 使用场景](figs/data_manage_sqlite_use_cases_diagram.png)

In [11]:
# 使用 SQLite 管理三张结构化表
# 为了演示主键约束，这里先显式建表，再写入数据。

db_path = DATA_OUTPUTS / 'finance_demo.sqlite'
if db_path.exists():
    db_path.unlink()

with sqlite3.connect(db_path) as conn:
    cur = conn.cursor()

    # 公司基本信息表：主键为 stock_code
    cur.execute("""
        CREATE TABLE basic_info (
            stock_code TEXT PRIMARY KEY,
            firm_name TEXT,
            industry_name TEXT
        )
    """)

    # 年度财务指标表：复合主键为 stock_code, year
    cur.execute("""
        CREATE TABLE fin_ratio (
            stock_code TEXT NOT NULL,
            year INTEGER NOT NULL,
            roe REAL,
            leverage REAL,
            PRIMARY KEY (stock_code, year)
        )
    """)

    # 年度收益率特征表：复合主键为 stock_code, year
    cur.execute("""
        CREATE TABLE ret_yearly (
            stock_code TEXT NOT NULL,
            year INTEGER NOT NULL,
            ret_mean REAL,
            ret_sd REAL,
            turnover_mean REAL,
            n_trade_days INTEGER,
            PRIMARY KEY (stock_code, year)
        )
    """)

    # 由于表已经建好，这里用 append 插入数据。
    basic_info.to_sql('basic_info', conn, if_exists='append', index=False)
    fin_ratio.to_sql('fin_ratio', conn, if_exists='append', index=False)
    ret_yearly.to_sql('ret_yearly', conn, if_exists='append', index=False)

print(f'SQLite 数据库已保存：{db_path}')

SQLite 数据库已保存：_demo_data_manage/data/outputs/finance_demo.sqlite


In [12]:
# 查看 SQLite 文件中的表

with sqlite3.connect(db_path) as conn:
    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        conn
    )

display(tables)

,name
0,basic_info
1,fin_ratio
2,ret_yearly


In [13]:
# 用 SQL 构造与 analysis_base 对应的 firm-year 分析底表

query = """
    SELECT
        f.stock_code,
        b.firm_name,
        b.industry_name,
        f.year,
        f.roe,
        f.leverage,
        r.ret_mean,
        r.ret_sd,
        r.turnover_mean,
        r.n_trade_days
    FROM fin_ratio AS f
    LEFT JOIN ret_yearly AS r
        ON f.stock_code = r.stock_code
       AND f.year = r.year
    LEFT JOIN basic_info AS b
        ON f.stock_code = b.stock_code
    ORDER BY f.stock_code, f.year
"""

with sqlite3.connect(db_path) as conn:
    analysis_from_sql = pd.read_sql_query(query, conn)

display(analysis_from_sql)
check_key(analysis_from_sql, ['stock_code', 'year'], 'analysis_from_sql')

,stock_code,firm_name,industry_name,year,roe,leverage,ret_mean,ret_sd,turnover_mean,n_trade_days
0,000001,平安银行,银行,2022,0.098,0.915,NaN,NaN,NaN,NaN
1,000001,平安银行,银行,2023,0.105,0.918,0.004000,0.007211,0.021333,3.0
2,000002,万科A,房地产,2022,0.072,0.772,NaN,NaN,NaN,NaN
3,000002,万科A,房地产,2023,0.068,0.781,0.003000,0.005000,0.016000,3.0
4,000003,国农科技,医药生物,2022,0.041,0.432,NaN,NaN,NaN,NaN
5,000003,国农科技,医药生物,2023,0.055,0.446,0.002667,0.008021,0.012000,3.0


analysis_from_sql 的行数：6
analysis_from_sql 的唯一主键数：6
analysis_from_sql 是否存在重复主键：False
--------------------------------------------------


这段 SQL 与前面的 `pandas.merge()` 做的是同一件事：以 `fin_ratio` 为主表，把已经聚合好的 `ret_yearly` 和公司基本信息 `basic_info` 并入进来。

区别在于，SQL 把表之间的关系写得更显式。对于表数量较多、查询需要反复修改的项目，SQLite 会比多次手工 `merge()` 更清楚。

In [14]:
# 参数化查询：避免把条件直接拼进 SQL 字符串
# 这里查询指定行业的 firm-year 观测。

industry = '银行'

query_by_industry = """
    SELECT
        f.stock_code,
        b.firm_name,
        b.industry_name,
        f.year,
        f.roe,
        f.leverage
    FROM fin_ratio AS f
    LEFT JOIN basic_info AS b
        ON f.stock_code = b.stock_code
    WHERE b.industry_name = ?
    ORDER BY f.year
"""

with sqlite3.connect(db_path) as conn:
    bank_data = pd.read_sql_query(query_by_industry, conn, params=(industry,))

display(bank_data)

,stock_code,firm_name,industry_name,year,roe,leverage
0,000001,平安银行,银行,2022,0.098,0.915
1,000001,平安银行,银行,2023,0.105,0.918


In [15]:
# 把分析结果写回 SQLite
# 示例：按行业计算平均 ROE。

industry_summary = (
    analysis_from_sql
    .groupby('industry_name', as_index=False)
    .agg(
        roe_mean=('roe', 'mean'),
        leverage_mean=('leverage', 'mean'),
        n_obs=('roe', 'size')
    )
)

with sqlite3.connect(db_path) as conn:
    industry_summary.to_sql('industry_summary', conn, if_exists='replace', index=False)

print('行业汇总结果：')
display(industry_summary)

行业汇总结果：

,industry_name,roe_mean,leverage_mean,n_obs
0,医药生物,0.0480,0.4390,2
1,房地产,0.0700,0.7765,2
2,银行,0.1015,0.9165,2


## DuckDB：直接查询本地数据文件

DuckDB 更偏向分析型 SQL 引擎。它的一个常用场景是：数据已经以 Parquet 或 CSV 文件形式保存在本地，不想先导入数据库，就希望直接执行 SQL 查询。

可以粗略区分三者的位置：

| 工具 | 更适合的任务 |
|---|---|
| Parquet | 保存清洗后的标准化分析数据 |
| SQLite | 管理多张结构化关系表 |
| DuckDB | 直接查询本地 Parquet / CSV 文件，并做分析型 SQL |

如果你在本地处理多个 Parquet 文件，DuckDB 往往很方便。若本地还没有安装，可以运行：

```bash
pip install duckdb
```

In [16]:
# DuckDB 示例：直接查询本地文件
# 如果没有安装 duckdb，代码会给出提示而不会中断。

try:
    import duckdb
    has_duckdb = True
except ImportError:
    has_duckdb = False
    print('当前环境没有安装 duckdb。如需运行本节示例，请先执行：pip install duckdb')

if has_duckdb:
    if has_parquet:
        duck_query = f"""
            SELECT
                industry_name,
                COUNT(*) AS n_obs,
                AVG(roe) AS roe_mean,
                AVG(leverage) AS leverage_mean
            FROM read_parquet('{parquet_path.as_posix()}')
            GROUP BY industry_name
            ORDER BY industry_name
        """
    else:
        duck_query = f"""
            SELECT
                industry_name,
                COUNT(*) AS n_obs,
                AVG(roe) AS roe_mean,
                AVG(leverage) AS leverage_mean
            FROM read_csv_auto('{csv_path.as_posix()}')
            GROUP BY industry_name
            ORDER BY industry_name
        """

    duck_result = duckdb.sql(duck_query).df()
    display(duck_result)

当前环境没有安装 duckdb。如需运行本节示例，请先执行：pip install duckdb


## 一个可复用的数据管理工作流

到这里，本章已经完成了从多张源表到分析底表，再到存储和查询的完整流程。可以把它固化成如下步骤：

1. **Step 1**: 明确研究问题和目标粒度，例如 `firm-year`；
2. **Step 2**: 为每张原始表写清楚粒度、主键、来源和时间范围；
3. **Step 3**: 对不能直接合并的高频表先聚合或转换；
4. **Step 4**: 构造 `analysis_base`，并检查目标主键是否仍然唯一；
5. **Step 5**: 将标准化数据保存为 Parquet 或 CSV；
6. **Step 6**: 对多表关系较复杂的项目，使用 SQLite 或 DuckDB 查询；
7. **Step 7**: 保留 README、数据字典和处理脚本，使结果可复现。

![数据管理端到端工作流](figs/data_manage_end_to_end_workflow.png)

In [17]:
def build_analysis_base(fin_ratio, daily_ret, basic_info):
    """
    从年度财务指标、日度收益率和公司基本信息构造 firm-year 分析底表。

    这段函数把本章前半部分的处理逻辑固化下来：
    1. 检查各原始表主键；
    2. 将日度收益率聚合到 firm-year；
    3. 合并为 analysis_base；
    4. 再次检查目标主键。
    """
    check_key(fin_ratio, ['stock_code', 'year'], 'fin_ratio')
    check_key(daily_ret, ['stock_code', 'trade_date'], 'daily_ret')
    check_key(basic_info, ['stock_code'], 'basic_info')

    ret_yearly = daily_ret.copy()
    ret_yearly['year'] = ret_yearly['trade_date'].dt.year
    ret_yearly = (
        ret_yearly
        .groupby(['stock_code', 'year'], as_index=False)
        .agg(
            ret_mean=('ret', 'mean'),
            ret_sd=('ret', 'std'),
            turnover_mean=('turnover', 'mean'),
            n_trade_days=('ret', 'size')
        )
    )

    analysis_base = (
        fin_ratio
        .merge(ret_yearly, on=['stock_code', 'year'], how='left')
        .merge(basic_info, on='stock_code', how='left')
    )

    check_key(analysis_base, ['stock_code', 'year'], 'analysis_base')
    return analysis_base

analysis_base_2 = build_analysis_base(fin_ratio, daily_ret, basic_info)
display(analysis_base_2)

fin_ratio 的行数：6
fin_ratio 的唯一主键数：6
fin_ratio 是否存在重复主键：False
--------------------------------------------------
daily_ret 的行数：9
daily_ret 的唯一主键数：9
daily_ret 是否存在重复主键：False
--------------------------------------------------
basic_info 的行数：3
basic_info 的唯一主键数：3
basic_info 是否存在重复主键：False
--------------------------------------------------
analysis_base 的行数：6
analysis_base 的唯一主键数：6
analysis_base 是否存在重复主键：False
--------------------------------------------------


,stock_code,year,roe,leverage,ret_mean,ret_sd,turnover_mean,n_trade_days,firm_name,industry_name
0,000001,2022,0.098,0.915,NaN,NaN,NaN,NaN,平安银行,银行
1,000001,2023,0.105,0.918,0.004000,0.007211,0.021333,3.0,平安银行,银行
2,000002,2022,0.072,0.772,NaN,NaN,NaN,NaN,万科A,房地产
3,000002,2023,0.068,0.781,0.003000,0.005000,0.016000,3.0,万科A,房地产
4,000003,2022,0.041,0.432,NaN,NaN,NaN,NaN,国农科技,医药生物
5,000003,2023,0.055,0.446,0.002667,0.008021,0.012000,3.0,国农科技,医药生物


## 检查清单

后续处理真实数据时，可以把下面这组问题作为最小检查清单：

- 每张表是否有明确主键？
- 主键是否真的唯一？
- 最终分析底表的目标粒度是什么？
- 高频表是否已经先聚合到目标粒度？
- 每次合并后，目标主键是否仍然唯一？
- 重要中间结果是否保存为可复用格式？
- README 或数据字典是否说明了数据来源、主键、粒度和更新时间？

这组检查不复杂，但能避免很多「代码正常运行、结果悄悄出错」的问题。

## 提示词模式：让 AI 帮你检查数据表关系

当项目中有很多张表时，可以把表结构告诉 AI，让它先帮你检查合并逻辑。提示词可以这样写：

```text
我正在构造一张 firm-year 层面的分析底表。现在有如下数据表：

- basic_info：公司基本信息，粒度为 firm，主键为 stock_code
- fin_ratio：年度财务指标，粒度为 firm-year，主键为 (stock_code, year)
- daily_ret：日度收益率，粒度为 firm-date，主键为 (stock_code, trade_date)
- macro_rate：宏观利率，粒度为 month，主键为 month
- ann_meta：公告元数据，粒度为 firm-date-document，主键为 (stock_code, ann_date, doc_id)

请你帮我完成以下任务：

1. 判断每张表能否直接合并到 firm-year 分析底表；
2. 说明不能直接合并的原因；
3. 给出需要先聚合或转换的变量；
4. 设计从原始表到 analysis_base 的处理流程；
5. 给出 Python 代码框架，并在每次合并后检查主键是否仍然唯一。
```

这里的关键是把粒度和主键说清楚。只说「我有几张表，请帮我合并」，通常得不到可靠结果。

## 本章小结

本章的核心不是学习某一个函数，而是形成一套数据管理判断：

- 多来源金融数据的主要困难，不是读入文件，而是不同表具有不同主键和粒度；
- 构造分析底表时，应先确定目标粒度，再判断哪些表可以直接合并，哪些表必须先聚合；
- CSV、Parquet、SQLite 和 DuckDB 各有位置，选择工具之前要先想清楚数据表之间的关系；
- 对课程项目和论文复现而言，数据契约、目录结构、README 和处理脚本，都是可复现工作流的一部分。

简言之，本章解决的是一个完整问题：从多张原始数据表出发，如何合并、保存、查询，并形成可复用的数据工作流。